In [7]:
import sys
import torch
import torch.nn as nn

# 将 src 目录加入 Python 搜索路径，便于在 notebook 中直接导入模块
sys.path.append("../src")

from embedding import TransformerEmbedding
from encoder import Encoder
from decoder import Decoder
from attention import MultiHeadAttention
from layernorm import LayerNorm

In [8]:
class Transformer(nn.Module):
    def __init__(
        self,
        src_pad_idx,
        tgt_pad_idx,
        src_voc_size,
        tgt_voc_size,
        d_model,
        max_len,
        n_head,
        n_layer,
        ffn_hidden_size,
        drop_prob,
        device,
    ):
        super().__init__()

        # 编码器负责源序列建模
        self.encoder = Encoder(
            src_voc_size, max_len, d_model, ffn_hidden_size, n_head, n_layer, drop_prob, device
        )

        # 解码器负责目标序列建模与交叉注意力
        self.decoder = Decoder(
            tgt_voc_size, max_len, d_model, ffn_hidden_size, n_head, n_layer, drop_prob, device
        )

        self.src_pad_idx = src_pad_idx
        self.tgt_pad_idx = tgt_pad_idx
        self.device = device

    def make_pad_mask(self, q, k, pad_idx_q, pad_idx_k):
        """
        构造 padding mask。
        返回形状: [batch_size, 1, len_q, len_k]
        True 表示可见，False 表示需要被 mask 掉。
        """
        # [batch, len_q] -> [batch, 1, len_q, 1]
        q_mask = q.ne(pad_idx_q).unsqueeze(1).unsqueeze(3)

        # [batch, len_k] -> [batch, 1, 1, len_k]
        k_mask = k.ne(pad_idx_k).unsqueeze(1).unsqueeze(2)

        # 广播后得到 [batch, 1, len_q, len_k]
        return q_mask & k_mask

    def make_causal_mask(self, q, k):
        """
        构造因果 mask（下三角），用于 decoder 自注意力，
        保证当前位置只能看到自己和之前的 token。
        返回形状: [1, 1, len_q, len_k]
        """
        len_q, len_k = q.size(1), k.size(1)
        causal = torch.tril(torch.ones(len_q, len_k, dtype=torch.bool, device=self.device))
        return causal.unsqueeze(0).unsqueeze(0)

    def forward(self, src, tgt):
        # 1) 编码器自注意力 mask（源序列内部）
        src_mask = self.make_pad_mask(src, src, self.src_pad_idx, self.src_pad_idx)

        # 2) 解码器自注意力 mask = padding mask ∩ causal mask
        tgt_pad_mask = self.make_pad_mask(tgt, tgt, self.tgt_pad_idx, self.tgt_pad_idx)
        tgt_causal_mask = self.make_causal_mask(tgt, tgt)
        tgt_mask = tgt_pad_mask & tgt_causal_mask

        # 3) 解码器-编码器交叉注意力 mask（query 来自 tgt，key 来自 src）
        src_tgt_mask = self.make_pad_mask(tgt, src, self.tgt_pad_idx, self.src_pad_idx)

        # 4) 编码器输出
        enc_src = self.encoder(src, src_mask)

        # 5) 为了保证 notebook 可独立运行，这里显式展开 decoder 的前向过程
        #    (与 src 目录中的模块保持一致，但不依赖 decoder.forward 内部实现细节)
        dec_out = self.decoder.embedding(tgt)
        for layer in self.decoder.layers:
            dec_out = layer(enc_src, dec_out, tgt_mask, src_tgt_mask)

        # 6) 投影到目标词表维度，得到 logits
        out = self.decoder.fc(dec_out)
        return out

In [9]:
# 形状自检示例（可运行）
device = "cpu"
model = Transformer(
    src_pad_idx=0,
    tgt_pad_idx=0,
    src_voc_size=1000,
    tgt_voc_size=1200,
    d_model=128,
    max_len=64,
    n_head=8,
    n_layer=2,
    ffn_hidden_size=512,
    drop_prob=0.1,
    device=device,
).to(device)

# batch=2, src_len=6, tgt_len=5
src = torch.tensor([[2, 7, 9, 0, 0, 0], [3, 8, 1, 4, 0, 0]], device=device)
tgt = torch.tensor([[1, 5, 6, 0, 0], [1, 10, 11, 12, 0]], device=device)

out = model(src, tgt)
print("output shape:", out.shape)

output shape: torch.Size([2, 5, 1200])


## Transformer 前向传播说明

本实现中一共用到 3 种 mask：

- `src_mask`：编码器自注意力的 padding mask。
- `tgt_mask`：解码器自注意力 mask，等于 `tgt_pad_mask & causal_mask`。
- `src_tgt_mask`：解码器交叉注意力 mask。

常见形状约定：

- `src`: `[batch, src_len]`
- `tgt`: `[batch, tgt_len]`
- `mask`: `[batch, 1, q_len, k_len]`

这样可以和多头注意力中的 `score` 形状 `[batch, head, q_len, k_len]` 对齐并广播。